# 🐔🦆 Train YOLOv8 — Nhận diện Gà & Vịt
> Chạy từng cell theo thứ tự từ trên xuống

**Trước khi chạy:** Vào `Runtime` → `Change runtime type` → chọn **T4 GPU**

## 📂 Bước 3 — Kết nối Google Drive (để lưu model sau khi train)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
# Tạo thư mục lưu project trên Drive
PROJECT_DIR = '/content/drive/MyDrive/ga-vit-detection'
os.makedirs(PROJECT_DIR, exist_ok=True)
print(f'Project dir: {PROJECT_DIR}')

## ⚗️ Bước 6 — Augment dataset (tăng 3x)

In [ ]:
import cv2
import numpy as np
import albumentations as A
from pathlib import Path
from tqdm import tqdm

IMG_SIZE = 640
AUG_PER_IMAGE = 3

transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.08, scale_limit=0.25, rotate_limit=20,
                       border_mode=cv2.BORDER_REFLECT_101, p=0.6),
    A.OneOf([
        A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3),
        A.RandomGamma(gamma_limit=(70, 140)),
    ], p=0.7),
    A.OneOf([
        A.HueSaturationValue(hue_shift_limit=15, sat_shift_limit=40, val_shift_limit=30),
        A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.3, hue=0.1),
    ], p=0.6),
    A.OneOf([
        A.GaussianBlur(blur_limit=(3, 5)),
        A.MotionBlur(blur_limit=5),
    ], p=0.2),
    A.RandomShadow(p=0.15),
], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels'],
                             min_visibility=0.3, min_area=100))

IMG_DIR = MERGED / 'images' / 'train'
LBL_DIR = MERGED / 'labels' / 'train'

originals = [p for p in IMG_DIR.glob('*') if '_aug' not in p.stem
             and p.suffix.lower() in {'.jpg','.jpeg','.png'}]
print(f'Augmenting {len(originals)} ảnh gốc × {AUG_PER_IMAGE}...')

created = 0
for img_path in tqdm(originals):
    lbl_path = LBL_DIR / (img_path.stem + '.txt')
    if not lbl_path.exists(): continue

    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(cv2.resize(img, (IMG_SIZE, IMG_SIZE)), cv2.COLOR_BGR2RGB)

    class_ids, bboxes = [], []
    with open(lbl_path) as f:
        for line in f:
            p = line.strip().split()
            if len(p) == 5:
                class_ids.append(int(p[0]))
                bboxes.append([max(0.001, min(0.999, float(x))) for x in p[1:]])

    for idx in range(AUG_PER_IMAGE):
        try:
            out = transform(image=img, bboxes=bboxes, class_labels=class_ids)
            if not out['bboxes']: continue
            stem = f'{img_path.stem}_aug{idx:02d}'
            cv2.imwrite(str(IMG_DIR / (stem + '.jpg')),
                        cv2.cvtColor(out['image'], cv2.COLOR_RGB2BGR),
                        [cv2.IMWRITE_JPEG_QUALITY, 95])
            with open(LBL_DIR / (stem + '.txt'), 'w') as f:
                for cls, (x,y,w,h) in zip(out['class_labels'], out['bboxes']):
                    f.write(f'{cls} {x:.6f} {y:.6f} {w:.6f} {h:.6f}\n')
            created += 1
        except: pass

total_after = len(list(IMG_DIR.glob('*')))
print(f'Tạo thêm {created} ảnh aug → Tổng train: {total_after} ảnh')

## ⚙️ Bước 7 — Tạo file data.yaml

In [ ]:
yaml_content = f"""path: /content/dataset
train: images/train
val:   images/val

nc: 2
names:
  0: ga
  1: vit
"""

with open('/content/data.yaml', 'w') as f:
    f.write(yaml_content)

print('data.yaml đã tạo:')
print(yaml_content)

## 🚀 Bước 8 — TRAIN MODEL

> ⏱️ Ước tính: ~30–60 phút với GPU T4 trên Colab

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8m.pt')  # tải pretrained weights tự động

results = model.train(
    data='/content/data.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,
    device=0,              # GPU

    # Augmentation
    flipud=0.3,
    fliplr=0.5,
    degrees=15,
    mosaic=1.0,
    mixup=0.1,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,

    project='/content/drive/MyDrive/ga-vit-detection/runs',  # lưu thẳng lên Drive
    name='ga_vit_v1',
    save=True,
    plots=True,
)

print('\n✅ Train xong!')
print('Model tốt nhất:', results.save_dir + '/weights/best.pt')

## 📊 Bước 9 — Đánh giá model

In [ ]:
from ultralytics import YOLO

# Load model vừa train
best_path = '/content/drive/MyDrive/ga-vit-detection/runs/ga_vit_v1/weights/best.pt'
model = YOLO(best_path)

metrics = model.val(data='/content/data.yaml', conf=0.45, iou=0.5, plots=True)

print('\n📊 Kết quả đánh giá:')
print(f'  mAP50:        {metrics.box.map50:.3f}  ← mục tiêu > 0.85')
print(f'  mAP50-95:     {metrics.box.map:.3f}')
print(f'  Precision Gà: {metrics.box.p[0]:.3f}')
print(f'  Precision Vịt:{metrics.box.p[1]:.3f}')
print(f'  Recall Gà:    {metrics.box.r[0]:.3f}')
print(f'  Recall Vịt:   {metrics.box.r[1]:.3f}')

if metrics.box.map50 >= 0.85:
    print('\n✅ Đạt mục tiêu! Chuyển sang detect.py')
else:
    print('\n⚠️  Chưa đạt — xem confusion matrix, bổ sung data yếu rồi train lại')

## 🖼️ Bước 10 — Xem kết quả trực quan

In [ ]:
from IPython.display import Image, display
import glob, os

run_dir = '/content/drive/MyDrive/ga-vit-detection/runs/ga_vit_v1'

# Hiển thị confusion matrix
cm_path = os.path.join(run_dir, 'confusion_matrix_normalized.png')
if os.path.exists(cm_path):
    print('Confusion Matrix:')
    display(Image(cm_path, width=500))

# Hiển thị loss curve
results_path = os.path.join(run_dir, 'results.png')
if os.path.exists(results_path):
    print('\nLoss & Metrics curves:')
    display(Image(results_path, width=700))

# Xem ảnh dự đoán mẫu
val_imgs = glob.glob(os.path.join(run_dir, 'val_batch*.jpg'))[:1]
for p in val_imgs:
    print('\nMẫu dự đoán trên val set:')
    display(Image(p, width=700))

## 🎬 Bước 11 — Test nhận diện trên ảnh/video

> Upload 1 ảnh hoặc video gà/vịt để test nhanh

In [ ]:
from google.colab import files
from ultralytics import YOLO
from IPython.display import Image, display

# Upload file test
uploaded = files.upload()
test_file = list(uploaded.keys())[0]
print(f'File test: {test_file}')

# Chạy predict
model = YOLO('/content/drive/MyDrive/ga-vit-detection/runs/ga_vit_v1/weights/best.pt')
results = model.predict(
    source=test_file,
    conf=0.45,
    iou=0.5,
    save=True,
    project='/content/test_results',
    name='predict'
)

# Hiển thị kết quả
for r in results:
    print(f'Phát hiện: {len(r.boxes)} đối tượng')
    for box in r.boxes:
        cls = int(box.cls[0])
        conf = float(box.conf[0])
        name = ['Gà', 'Vịt'][cls]
        print(f'  → {name}: {conf:.2f}')

# Xem ảnh kết quả
import glob
out_imgs = glob.glob('/content/test_results/predict/*.jpg')
for p in out_imgs[:1]:
    display(Image(p, width=640))

---
## ✅ Xong!
- Model đã được lưu tại: `Google Drive/ga-vit-detection/runs/ga_vit_v1/weights/best.pt`
- Tải `best.pt` về máy → chạy `src/detect.py` để test real-time trên webcam/video
- Nếu mAP50 < 0.85 → xem confusion matrix → bổ sung data → chạy lại từ Bước 6